<a href="https://colab.research.google.com/github/simongalaxy/sentimental_agent/blob/master/Amazon_Reviews_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#! pip install sentence-transformers transformers bertopic

In [2]:
import pandas as pd
from typing import List, Dict
from pprint import pformat

In [3]:
filePath = "/content/amazon_google_reviews.csv"

In [4]:
def load_clean_split_data_from_csv(path: str) -> List[pd.DataFrame]:

  # load data from csv file.
  df = pd.read_csv(path)
  print(df.info())

  # clean data.
  str_cols = ["Platform", "Product/Location_Name", "Review_Text"]
  df[str_cols] = df[str_cols].apply(lambda x: x.str.strip())

  # split data into a list of dataframes by platforms and products.
  dfs = []
  for platform in df["Platform"].unique():
    for product in df[df["Platform"] == platform]["Product/Location_Name"].unique():
      df_temp = df[(df["Platform"] == platform) & (df["Product/Location_Name"] == product)].reset_index()
      dfs.append(df_temp)

  print(f"Total {len(dfs)} dataframes splitted.")

  return dfs

In [5]:
dfs = load_clean_split_data_from_csv(path = filePath)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6200 entries, 0 to 6199
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Review_ID              6200 non-null   object
 1   Platform               6200 non-null   object
 2   Product/Location_Name  6200 non-null   object
 3   Review_Text            6200 non-null   object
 4   Rating                 6200 non-null   int64 
 5   Date                   6200 non-null   object
 6   Helpful_Votes          6200 non-null   int64 
dtypes: int64(2), object(5)
memory usage: 339.2+ KB
None
Total 25 dataframes splitted.


In [6]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import paraphrase_mining

# --- Assumed Setup ---
# df = pd.DataFrame({"Review_Text": [...]})
class DuplicatesRemover:
  # 1. Load a lightweight, high-performance embedding model
  def __init__(self):
    self.model_name = "all-MiniLM-L6-v2"
    self.model = SentenceTransformer(self.model_name)

  def _group_semantic_similar_reviews(self, df: pd.DataFrame) -> pd.DataFrame:
    # 1. Extract reviews to a list
    sentences = df["Review_Text"].tolist()

    # 2. Perform Paraphrase Mining
    # (Removed the invalid 'min_score' argument)
    pairs = paraphrase_mining(self.model, sentences)

    # 3. Group the sentences using a Union-Find (Connected Components) structure
    parent = {i: i for i in range(len(sentences))}

    def find(i):
        if parent[i] == i:
            return i
        parent[i] = find(parent[i])  # Path compression
        return parent[i]

    def union(i, j):
        root_i = find(i)
        root_j = find(j)
        if root_i != root_j:
            parent[root_j] = root_i

    # Merge sentences ONLY if they meet or exceed the similarity threshold
    threshold = 0.85
    for score, i, j in pairs:
        if score >= threshold:
            union(i, j)

    # 4. Map the cluster root IDs directly back to your DataFrame
    # This gives every review in df a Group ID based on its semantic cluster root
    df["Semantic_Group_ID"] = [find(i) for i in range(len(sentences))]

    # 5. Extract the groupings for visual confirmation
    groupings = {}
    for i in range(len(sentences)):
        root = find(i)
        if root not in groupings:
            groupings[root] = []
        groupings[root].append(sentences[i])

    # 6. Print the Results
    print("--- SEMANTIC GROUPINGS ---")
    deduplicated_sentences = []

    for group_id, group in groupings.items():
        print(f"\nGroup {group_id}:")
        for sentence in group:
            print(f" - {sentence}")

        # The representative sentence of this group is the first one
        deduplicated_sentences.append(group[0])

    print("\n--- FINAL DEDUPLICATED LIST ---")
    for sent in deduplicated_sentences:
        print(f"✔ {sent}")

    return df



  def _consolidate_reviews_by_groups(self, df: pd.DataFrame) -> pd.DataFrame:
    # consolidate the df by Semantic_Group_ID.
    df_grouped = df.groupby(by=["Platform", "Product/Location_Name", "Semantic_Group_ID"]).agg(Count=("Review_ID", "count"), Average_Rating=("Rating", "mean"), Helpful_Votes=("Helpful_Votes", "sum")).reset_index()

    # get the first sentences by Semantic_Group_IDs.
    dfs_first_sentence = []
    for id in df["Semantic_Group_ID"].unique():
      temp = df[df["Semantic_Group_ID"] == id][["Semantic_Group_ID", "Review_Text"]].head(1)
      dfs_first_sentence.append(temp)

    df_first_sentence = pd.concat(dfs_first_sentence)

    # merge these 2 dataframes as output.
    df_unique_review = pd.merge(df_grouped, df_first_sentence, on="Semantic_Group_ID")

    return df_unique_review


  def _save_unique_reviews_to_csv(self, dfs: List[pd.DataFrame]) -> None:
    for df in dfs:
      platform = df["Platform"].to_list()[0].replace(" ", "_")
      product = df["Product/Location_Name"].to_list()[0].replace(" ", "_")
      filename = f"{platform}-{product}-unique_review.csv"
      df[["Platform", "Product/Location_Name", "Semantic_Group_ID", "Review_Text", "Count", "Average_Rating", "Helpful_Votes"]].sort_values(by="Review_Text", ascending=True).to_csv(filename)

    return

  def remove_duplicate_reviews(self, dfs: List[pd.DataFrame]) -> List[pd.DataFrame]:
    dfs_processed = [self._group_semantic_similar_reviews(df=df) for df in dfs]
    dfs_unique_review = [self._consolidate_reviews_by_groups(df=df) for df in dfs_processed]
    self._save_unique_reviews_to_csv(dfs=dfs_unique_review)

    return dfs_unique_review



In [7]:
remover = DuplicatesRemover()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
dfs_unique_review = remover.remove_duplicate_reviews(dfs=dfs)

Streaming output truncated to the last 5000 lines.
Group 60:
 - Honestly, absolute perfection! easily the best purchase i have made this year. high quality materials, elegant design, and works flawlessly.
 - Absolute perfection! Easily the best purchase I have made this year. High quality materials, elegant design, and works flawlessly. #verifiedpurchase
 - Summary: Absolute perfection! Easily the best purchase I have made this year. High quality materials, elegant design, and works flawlessly.
 - Bought this recently. Absolute perfection! Easily the best purchase I have made this year. High quality materials, elegant design, and works flawlessly.
 - Honestly, absolute perfection! easily the best purchase i have made this year. high quality materials, elegant design, and works flawlessly.
 - Absolute perfection! Easily the best purchase I have made this year. High quality materials, elegant design, and works flawlessly. #verifiedpurchase
 - Absolute perfection! Easily the best purchase

In [9]:
dfs_unique_review[0].head(10)

,Platform,Product/Location_Name,Semantic_Group_ID,Count,Average_Rating,Helpful_Votes,Review_Text
0,Google Locations,Metropolitan Dental Health Clinic,0,2,1.0,2,Terrible hygiene and filthy tables. Saw flies ...
1,Google Locations,Metropolitan Dental Health Clinic,2,10,4.0,11,Wonderful experience overall. The staff was co...
2,Google Locations,Metropolitan Dental Health Clinic,3,4,2.0,8,Mediocre stay. Cleanliness was questionable an...
3,Google Locations,Metropolitan Dental Health Clinic,6,11,5.0,66,"Incredible place to visit! Breathtaking flora,..."
4,Google Locations,Metropolitan Dental Health Clinic,7,7,4.0,11,Great atmosphere and delicious food! The pasta...
5,Google Locations,Metropolitan Dental Health Clinic,9,2,1.0,4,Rude management and hostile attitude towards c...
6,Google Locations,Metropolitan Dental Health Clinic,15,6,3.0,8,"Honestly, decent spot, nothing extraordinary. ..."
7,Google Locations,Metropolitan Dental Health Clinic,18,3,1.0,39,"Honestly, rude management and hostile attitude..."
8,Google Locations,Metropolitan Dental Health Clinic,19,5,3.0,23,Fair experience. Room was adequately clean and...
9,Google Locations,Metropolitan Dental Health Clinic,20,2,1.0,5,Bought this recently. Total rip-off. Paid prem...


In [ ]:
#! pip install bertopic

In [29]:
from transformers import pipeline

class ReviewsClassifier:
  def __init__(self):
    # sentimental classifier settings.
    self.sentimental_model = "distilbert-base-uncased-finetuned-sst-2-english"
    self.sentimental_classifier = pipeline("sentiment-analysis", model=self.sentimental_model)

  # Batch process unique reviews
  def label_sentimental(self, dfs: List[pd.DataFrame], review_col: str) -> List[pd.DataFrame]:
    for i, df in enumerate(dfs):
      platform = df["Platform"].to_list()[0].replace(" ", "_")
      product = df["Product/Location_Name"].to_list()[0].replace(" ", "_")
      filename = f"{platform}-{product}-sentimental.csv"

      print(f"No. {i}/{len(dfs)}: Start label sentimental for dataframe - Platform: {platform}, Product: {product}")
      df['Sentiment'] = [res['label'] for res in self.sentimental_classifier(df[review_col].tolist(), batch_size=32)]
      df.to_csv(f"./sentimental/{filename}")
      print(f"No. {i}/{len(dfs)}: End label sentimental for dataframe - Platform: {platform}, Product: {product} and save as csv.")

    print(f"total {len(dfs)} dataframes labelled sentimentals.")

    return dfs



In [15]:
classifier = ReviewsClassifier()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [17]:
dfs_sentimental = classifier.label_sentimental(dfs=dfs_unique_review, review_col="Review_Text")

No. 0/25: Start label sentimental for dataframe - Platform: Google_Locations, Product: Metropolitan_Dental_Health_Clinic
No. 0/25: End label sentimental for dataframe - Platform: Google_Locations, Product: Metropolitan_Dental_Health_Clinic and save as csv.
No. 1/25: Start label sentimental for dataframe - Platform: Google_Locations, Product: Blue_Bottle_Artisan_Coffee_Bar
No. 1/25: End label sentimental for dataframe - Platform: Google_Locations, Product: Blue_Bottle_Artisan_Coffee_Bar and save as csv.
No. 2/25: Start label sentimental for dataframe - Platform: Google_Locations, Product: Bella_Italia_Trattoria
No. 2/25: End label sentimental for dataframe - Platform: Google_Locations, Product: Bella_Italia_Trattoria and save as csv.
No. 3/25: Start label sentimental for dataframe - Platform: Google_Locations, Product: Highland_Public_Library_&_Cultural_Center
No. 3/25: End label sentimental for dataframe - Platform: Google_Locations, Product: Highland_Public_Library_&_Cultural_Center a

In [20]:
dfs_sentimental[24].head(10)

,Platform,Product/Location_Name,Semantic_Group_ID,Count,Average_Rating,Helpful_Votes,Review_Text,Sentiment
0,Amazon,Anker 65W GaN Fast Charger,2,16,5.0,62,Review update after 3 months: Superb product! ...,POSITIVE
1,Amazon,Anker 65W GaN Fast Charger,3,3,2.0,5,"Honestly, looks nice on the desk, but function...",NEGATIVE
2,Amazon,Anker 65W GaN Fast Charger,4,13,5.0,103,Game changer! Saves me so much time and effort...,POSITIVE
3,Amazon,Anker 65W GaN Fast Charger,6,12,5.0,106,Review update after 3 months: Absolute perfect...,POSITIVE
4,Amazon,Anker 65W GaN Fast Charger,8,2,2.0,4,Disappointed with the build quality. It is oka...,NEGATIVE
5,Amazon,Anker 65W GaN Fast Charger,13,6,4.0,8,Impressive performance for the price. Clear so...,POSITIVE
6,Amazon,Anker 65W GaN Fast Charger,15,3,2.0,6,Review update after 3 months: Subpar durabilit...,NEGATIVE
7,Amazon,Anker 65W GaN Fast Charger,16,6,1.0,36,Review update after 3 months: Do not buy this!...,NEGATIVE
8,Amazon,Anker 65W GaN Fast Charger,17,9,4.0,14,Bought this recently. Much better than my prev...,NEGATIVE
9,Amazon,Anker 65W GaN Fast Charger,18,17,5.0,152,Remarkable craftsmanship. Every detail feels t...,POSITIVE


In [42]:
from bertopic import BERTopic
from bertopic.representation import TextGeneration
from transformers import pipeline
from sentence_transformers import SentenceTransformer

class TopicIdentifier:
  def __init__(self):
    # 1. Use a Hugging Face model via SentenceTransformers to extract embeddings
    # This creates mathematical representations of the meanings behind the reviews
    self.embedding_model_name = "all-MiniLM-L6-v2"
    self.embedding_model = SentenceTransformer(self.embedding_model_name)

    # Load a Hugging Face text generation pipeline
    self.generator_model_name = 'google/flan-t5-base'
    self.generator = pipeline('text-generation', model=self.generator_model_name) #'text2text-generation'
    self.representation_model = TextGeneration(self.generator)

    # 2. Pass the Hugging Face model into BERTopic
    # BERTopic handles the UMAP reduction, HDBSCAN clustering, and c-TF-IDF keyword extraction
    # Pass it to BERTopic to label the discovered clusters automatically
    self.topic_model = BERTopic(
        embedding_model=self.embedding_model,
        representation_model=self.representation_model
    )

  # 3. Train the model on your data to automatically discover the categories
  def get_topics_from_reviews(self, reviews: List[str]):
    topics, probs = self.topic_model.fit_transform(reviews)

    # 4. Extract and print the discovered categories (topics)
    topic_info = self.topic_model.get_topic_info()
    print(f"Data_type of topic_info: {type(topic_info)}")
    # print(topic_info[["Topic", "Count", "Name"]])
    print(topic_info)

    return

In [43]:
identifier = TopicIdentifier()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BailingHybridForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BeamForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CohereCompassForC

In [44]:
test_df = dfs_sentimental[0]
# positive reviews.
positive_reviews = test_df[test_df["Sentiment"] == "POSITIVE"]["Review_Text"].tolist()

# negative reviews.
negative_reviews = test_df[test_df["Sentiment"] == "NEGATIVE"]["Review_Text"].tolist()


In [47]:
identifier.get_topics_from_reviews(reviews=negative_reviews)

Data_type of topic_info: <class 'pandas.core.frame.DataFrame'>
   Topic  Count    Name
0     -1     19  -1____
   Topic  Count    Name        Representation  \
0     -1     19  -1____  [, , , , , , , , , ]   

                                 Representative_Docs  
0  [Average food with very slow service. We waite...  
